# Notebook 05 — Truth Matching, Splits, and Supervision Table

**Responsibilities (plan §12.6):**
- Candidate-to-truth matching via one-to-one Hungarian assignment (§9.1)
- `candidate_to_truth_match.parquet` (§3.7)
- `splits.parquet` — spatial block cross-validation (§3.10, §10.2)
- `training_supervision_table.parquet` (§3.9)

**This notebook does NOT:**
- Run any detectors
- Compute features
- Train models
- Generate annotations

## Matching policy (§9.1)
- **Algorithm:** Hungarian (optimal one-to-one) with hard distance threshold `truth_match_radius_px`
- **UNCERTAIN halo exclusion:** candidates within `uncertain_halo_radius_px` of any UNCERTAIN annotation are excluded (not labeled negative)
- **Negatives** derived only from completed annotated crops
- **False negatives** (unmatched truth) tracked for evaluation, never trained on
- **Truth coordinates:** `refined_click_well_y/x_px` preferred; falls back to `raw_click_well_y/x_px` with `truth_coord_source` recorded

## Splits (§10.2)
- Spatial block CV at crop level (each crop = one block)
- Stratified by well_id: 60% train / 20% calibration / 20% test
- Deterministic hash-based assignment — reproducible across runs
- Reuses existing `splits.parquet` if present and registry version matches

## Supervision table (§3.9)
- Left-join of `mega_feature_table` + match results + splits
- **One row per union_id** — every union candidate gets a row
- `supervision_status` ∈ {`included`, `excluded`}; `target_binary` ∈ {0, 1, NaN}
- `sample_weight` computed via configurable strategy
- Candidates outside annotated territory → `supervision_status = excluded`, `target_binary = NaN`

## Diagnostics produced
- Per-crop match status breakdown with precision/recall proxies
- Matched-pair distance distribution
- Recall-at-radius sweep
- False-negative nearest-candidate distance analysis
- Diagnostic plots saved to `artifacts/reports/nb05_truth_matching/`
- Top features by Cohen's d separability
- QA overlays for worst-FN / worst-precision crops (when image paths resolvable)

In [ ]:
# -------------------------------------------------------------------
# REPO DISCOVERY
# -------------------------------------------------------------------
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / ".git").exists() or (p / "registries").exists():
            return p
    raise RuntimeError("Could not locate repo root from current working directory.")

REPO_ROOT = find_repo_root()
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
# -------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------
CFG = {
    # ── Execution / provenance ─────────────────────────────────────────────
    "execution_mode": "debug_small",
    "repo_root": str(REPO_ROOT),

    # ── Explicit path overrides (set to a concrete path string to bypass autodiscovery) ──
    "candidate_union_path_override": None,
    "mega_feature_table_path_override": None,
    "annotations_path_override": None,
    "crop_registry_path_override": None,
    "candidate_clusters_path_override": None,
    "candidate_union_membership_path_override": None,

    # ── Directory layout ───────────────────────────────────────────────────
    "tables_dir": "tables",
    "crop_registry_dir": "annotations/crop_registry",
    "clicked_truth_dir": "annotations/clicked_truth",
    "artifact_report_dir": "artifacts/reports/nb05_truth_matching",
    "manifest_dir": "artifacts/manifests",

    # ── Matching policy ────────────────────────────────────────────────────
    "matching_algorithm": "hungarian_thresholded",
    "matching_registry_version": "matching_v1_crop_hungarian_medoid_refined_halo",
    "truth_coordinate_policy": "refined_well_fallback_raw",
    "candidate_coordinate_policy": "union_medoid_well",
    "truth_match_radius_px": 8.0,

    # ── UNCERTAIN exclusion ────────────────────────────────────────────────
    # halo_exclusion: exclude ALL candidates within uncertain_halo_radius_px of any UNCERTAIN
    # assigned_only:  exclude only the single nearest candidate
    "uncertain_exclusion_policy": "halo_exclusion",
    "uncertain_halo_radius_px": 8.0,

    # ── Negative generation ────────────────────────────────────────────────
    "negative_generation_policy": "completed_annotated_crop_only",
    # annotator_status values that count as "annotation complete"
    "done_like_crop_statuses": ["done", "completed", "annotated"],
    # require at least one annotation to be present (guard against empty-done crops)
    "require_annotations_present_for_negative_generation": True,

    # ── Splits (§10.2) ────────────────────────────────────────────────────
    "split_train_frac": 0.60,
    "split_cal_frac": 0.20,
    "split_test_frac": 0.20,
    "split_seed": 42,
    "split_registry_version": "v1_nb05",
    # If True and a splits.parquet with matching split_registry_version exists, reuse it
    "reuse_existing_splits": True,

    # ── Sample weighting ──────────────────────────────────────────────────
    # "balanced"          : w = n_total / (2 * n_class)
    # "sqrt_inverse_freq" : w = 1/sqrt(class_freq), normalized to mean=1
    # "none"              : w = 1.0 for all supervised rows
    "sample_weight_strategy": "balanced",
    "positive_sample_weight_override": None,   # set to float to skip strategy
    "negative_sample_weight_override": None,
    "excluded_sample_weight": 0.0,

    # ── annotation_coverage_status values (from mega) treated as supervised territory ──
    "accepted_annotation_coverage_statuses_for_negatives": [
        "fully_annotated_crop", "annotated_crop", "complete", "in_crop"
    ],
    "use_mega_annotation_coverage_if_available": True,

    # ── Persistence flags ──────────────────────────────────────────────────
    "write_candidate_to_truth_match": True,
    "write_training_supervision_table": True,
    "write_truth_evaluation_summary": True,
    "write_splits": True,
    "write_reports": True,

    # ── QA / reporting ─────────────────────────────────────────────────────
    "qa_max_overlay_crops": 8,
    "qa_point_alpha": 0.9,
    "qa_fig_dpi": 160,
    "write_match_diagnostics": True,
    "min_annotations_per_crop_warn": 1,
}

CFG

In [ ]:
# -------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------
import gc
import hashlib
import json
import math
import os
import re
import time
from collections import defaultdict, Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import tifffile
import yaml

from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

print("imports ok")

In [ ]:
# -------------------------------------------------------------------
# LOGGING / IO / PROVENANCE HELPERS
# (self-contained; mirrors helpers in NB03/04 without reusing their state)
# -------------------------------------------------------------------
def ts_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def log(msg: str) -> None:
    print(f"[{ts_utc()}] {msg}", flush=True)

def ensure_dir(path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_run_id(prefix: str = "nb05") -> str:
    t = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    h = hashlib.sha1(f"{t}|{os.getpid()}|nb05".encode("utf-8")).hexdigest()[:8]
    return f"{prefix}_{t}_{h}"

def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode("utf-8")).hexdigest()

def safe_to_parquet(df: pd.DataFrame, path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
    except Exception as e:
        csv_path = path.with_suffix(".csv")
        log(f"[warn] parquet write failed ({e}); writing CSV fallback -> {csv_path.name}")
        df.to_csv(csv_path, index=False)
        return csv_path
    return path

def write_json(obj, path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=False), encoding="utf-8")
    return path

def read_yaml_or_json(path):
    path = Path(path)
    txt = path.read_text(encoding="utf-8")
    if path.suffix.lower() in {".yaml", ".yml"}:
        return yaml.safe_load(txt)
    return json.loads(txt)

def latest_matching_file(search_dir: Path, patterns: list) -> Path | None:
    """Return the most-recently-modified file matching any glob pattern in search_dir."""
    candidates = []
    if not search_dir.exists():
        return None
    for pat in patterns:
        candidates.extend(search_dir.glob(pat))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        return None
    candidates.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return candidates[0]

RUN_ID = make_run_id("nb05")

# TABLES_DIR / REPORT_DIR / MANIFEST_DIR: reuse from prior kernel (NB03/04) if present,
# otherwise derive from repo root. This lets NB05 run in the same kernel as NB03/04
# without redundant directory creation, while still working standalone.
try:
    _ = TABLES_DIR
    log(f"TABLES_DIR from prior kernel: {TABLES_DIR}")
except NameError:
    TABLES_DIR = ensure_dir(REPO_ROOT / CFG["tables_dir"])
    log(f"TABLES_DIR (from config): {TABLES_DIR}")

try:
    _ = ARTIFACT_DIR
    REPORT_DIR = ensure_dir(ARTIFACT_DIR / "reports" / "nb05_truth_matching")
    log(f"REPORT_DIR (via ARTIFACT_DIR from prior kernel): {REPORT_DIR}")
except NameError:
    REPORT_DIR = ensure_dir(REPO_ROOT / CFG["artifact_report_dir"])
    log(f"REPORT_DIR (from config): {REPORT_DIR}")

try:
    _ = MANIFEST_DIR
    log(f"MANIFEST_DIR from prior kernel: {MANIFEST_DIR}")
except NameError:
    MANIFEST_DIR = ensure_dir(REPO_ROOT / CFG["manifest_dir"])
    log(f"MANIFEST_DIR (from config): {MANIFEST_DIR}")

log(f"RUN_ID = {RUN_ID}")

In [ ]:
# -------------------------------------------------------------------
# DISCOVERY / IN-MEMORY RESOLUTION
#
# For each input table, the priority order is:
#   1. Explicit path override in CFG (always forces a disk reload)
#   2. Variable already in kernel scope from NB03/04 (use as-is, no reload)
#   3. Autodiscovery from disk (most-recently-modified matching parquet/yaml)
#
# The full pipeline (03_04_05) runs NB03→NB04→NB05 in one kernel so
# `candidate_union`, `mega`, `crop_registry` etc. are already live.
# The standalone path falls through to disk discovery.
#
# NB03/04 name the mega table `mega`; the standalone uses `mega_feature_table`.
# Both are checked below.
# -------------------------------------------------------------------
tables_dir        = REPO_ROOT / CFG["tables_dir"]
crop_registry_dir = REPO_ROOT / CFG["crop_registry_dir"]
clicked_truth_dir = REPO_ROOT / CFG["clicked_truth_dir"]

# ── Track source of each input for provenance ──────────────────────────────
_input_sources = {}

# ── candidate_union ───────────────────────────────────────────────────────
if CFG["candidate_union_path_override"]:
    candidate_union_path = Path(CFG["candidate_union_path_override"])
    _input_sources["candidate_union"] = f"override:{candidate_union_path}"
else:
    try:
        _ = candidate_union
        candidate_union_path = None
        _input_sources["candidate_union"] = "in_memory"
    except NameError:
        candidate_union_path = latest_matching_file(tables_dir, ["*candidate_union.parquet"])
        _input_sources["candidate_union"] = f"disk:{candidate_union_path}"

# ── mega_feature_table (NB03/04 kernel names it `mega`) ───────────────────
if CFG["mega_feature_table_path_override"]:
    mega_feature_table_path = Path(CFG["mega_feature_table_path_override"])
    _input_sources["mega_feature_table"] = f"override:{mega_feature_table_path}"
else:
    _mega_in_memory = False
    try:
        _ = mega_feature_table
        _mega_in_memory = True
    except NameError:
        pass
    if not _mega_in_memory:
        try:
            mega_feature_table = mega  # NB03/04 pipeline name
            _mega_in_memory = True
        except NameError:
            pass
    if _mega_in_memory:
        mega_feature_table_path = None
        _input_sources["mega_feature_table"] = "in_memory"
    else:
        mega_feature_table_path = latest_matching_file(tables_dir, ["*mega_feature_table.parquet"])
        _input_sources["mega_feature_table"] = f"disk:{mega_feature_table_path}"

# ── candidate_clusters ────────────────────────────────────────────────────
if CFG["candidate_clusters_path_override"]:
    candidate_clusters_path = Path(CFG["candidate_clusters_path_override"])
    _input_sources["candidate_clusters"] = f"override:{candidate_clusters_path}"
else:
    try:
        _ = candidate_clusters
        candidate_clusters_path = None
        _input_sources["candidate_clusters"] = "in_memory"
    except NameError:
        candidate_clusters_path = latest_matching_file(tables_dir, ["*candidate_clusters.parquet"])
        _input_sources["candidate_clusters"] = f"disk:{candidate_clusters_path}"

# ── candidate_union_membership ────────────────────────────────────────────
if CFG["candidate_union_membership_path_override"]:
    candidate_union_membership_path = Path(CFG["candidate_union_membership_path_override"])
    _input_sources["candidate_union_membership"] = f"override:{candidate_union_membership_path}"
else:
    try:
        _ = candidate_union_membership
        candidate_union_membership_path = None
        _input_sources["candidate_union_membership"] = "in_memory"
    except NameError:
        candidate_union_membership_path = latest_matching_file(tables_dir, ["*candidate_union_membership.parquet"])
        _input_sources["candidate_union_membership"] = f"disk:{candidate_union_membership_path}"

# ── annotations (NB02 writes tables/annotations.parquet as canonical) ────
if CFG["annotations_path_override"]:
    annotations_path = Path(CFG["annotations_path_override"])
    _input_sources["annotations"] = f"override:{annotations_path}"
else:
    # Annotations are never carried as a live variable by NB03/04, always from disk
    annotations_path = (
        tables_dir / "annotations.parquet"
        if (tables_dir / "annotations.parquet").exists()
        else latest_matching_file(clicked_truth_dir, ["*.parquet"])
    )
    _input_sources["annotations"] = f"disk:{annotations_path}"

# ── crop_registry (NB01/03 may have it in scope) ──────────────────────────
if CFG["crop_registry_path_override"]:
    crop_registry_path = Path(CFG["crop_registry_path_override"])
    _input_sources["crop_registry"] = f"override:{crop_registry_path}"
else:
    try:
        _ = crop_registry
        crop_registry_path = None
        _input_sources["crop_registry"] = "in_memory"
    except NameError:
        crop_registry_path = latest_matching_file(crop_registry_dir, ["*.parquet", "*.yaml", "*.yml"])
        _input_sources["crop_registry"] = f"disk:{crop_registry_path}"

# ── Summary ───────────────────────────────────────────────────────────────
INPUT_PATHS = {
    "candidate_union":            str(candidate_union_path) if candidate_union_path else "(in_memory)",
    "mega_feature_table":         str(mega_feature_table_path) if mega_feature_table_path else "(in_memory)",
    "candidate_clusters":         str(candidate_clusters_path) if candidate_clusters_path else "(in_memory)",
    "candidate_union_membership": str(candidate_union_membership_path) if candidate_union_membership_path else "(in_memory)",
    "annotations":                str(annotations_path) if annotations_path else None,
    "crop_registry":              str(crop_registry_path) if crop_registry_path else "(in_memory)",
}
log("Input resolution:")
for k, src in _input_sources.items():
    log(f"  {k}: {src}")

# annotations must always come from disk (NB03/04 don't carry it)
assert annotations_path is not None, (
    "No annotations file found. "
    "Expected tables/annotations.parquet (NB02 canonical) or a parquet in annotations/clicked_truth/. "
    "Run NB02 first."
)
# For disk-sourced tables, assert the files exist
for _name, _path in [
    ("candidate_union",      candidate_union_path),
    ("mega_feature_table",   mega_feature_table_path),
    ("crop_registry",        crop_registry_path),
]:
    if _path is not None:
        assert Path(_path).exists(), (
            f"Required input not found on disk: {_name} -> {_path}. "
            "Check path overrides or run upstream notebooks."
        )

In [ ]:
# -------------------------------------------------------------------
# LOAD INPUT TABLES
# Only reads from disk for tables not already resolved in-memory above.
# Tables already in kernel scope are used directly (no disk I/O).
# -------------------------------------------------------------------
log("Resolving input tables ...")
t0_load = time.perf_counter()

# candidate_union
if candidate_union_path is not None:
    candidate_union = pd.read_parquet(candidate_union_path)
    log(f"  candidate_union             : loaded from disk ({len(candidate_union):,} rows)")
else:
    # Make a copy so downstream mutations don't corrupt the NB03/04 table
    candidate_union = candidate_union.copy()
    log(f"  candidate_union             : from kernel ({len(candidate_union):,} rows)")

# mega_feature_table
if mega_feature_table_path is not None:
    mega_feature_table = pd.read_parquet(mega_feature_table_path)
    log(f"  mega_feature_table          : loaded from disk ({len(mega_feature_table):,} rows)")
else:
    mega_feature_table = mega_feature_table.copy()
    log(f"  mega_feature_table          : from kernel ({len(mega_feature_table):,} rows)")

# annotations — always from disk (NB03/04 don't carry this)
annotations = pd.read_parquet(annotations_path)
log(f"  annotations                 : loaded from disk ({len(annotations):,} rows)")

# candidate_clusters
if candidate_clusters_path is not None:
    candidate_clusters = (
        pd.read_parquet(candidate_clusters_path)
        if Path(candidate_clusters_path).exists()
        else pd.DataFrame()
    )
    log(f"  candidate_clusters          : loaded from disk ({len(candidate_clusters):,} rows)")
else:
    try:
        candidate_clusters = candidate_clusters.copy()
    except NameError:
        candidate_clusters = pd.DataFrame()
    log(f"  candidate_clusters          : from kernel ({len(candidate_clusters):,} rows)")

# candidate_union_membership
if candidate_union_membership_path is not None:
    candidate_union_membership = (
        pd.read_parquet(candidate_union_membership_path)
        if Path(candidate_union_membership_path).exists()
        else pd.DataFrame()
    )
    log(f"  candidate_union_membership  : loaded from disk ({len(candidate_union_membership):,} rows)")
else:
    try:
        candidate_union_membership = candidate_union_membership.copy()
    except NameError:
        candidate_union_membership = pd.DataFrame()
    log(f"  candidate_union_membership  : from kernel ({len(candidate_union_membership):,} rows)")

# crop_registry
if crop_registry_path is not None:
    _cr_ext = Path(crop_registry_path).suffix.lower()
    if _cr_ext == ".parquet":
        crop_registry = pd.read_parquet(crop_registry_path)
    elif _cr_ext in {".yaml", ".yml"}:
        _payload = read_yaml_or_json(crop_registry_path)
        crop_registry = pd.DataFrame(
            _payload["records"] if isinstance(_payload, dict) and "records" in _payload else _payload
        )
    else:
        raise ValueError(f"Unsupported crop registry extension: {_cr_ext}")
    log(f"  crop_registry               : loaded from disk ({len(crop_registry):,} rows)")
else:
    crop_registry = crop_registry.copy()
    log(f"  crop_registry               : from kernel ({len(crop_registry):,} rows)")

log(f"  resolved in {time.perf_counter() - t0_load:.2f}s")

In [ ]:
# -------------------------------------------------------------------
# CONTRACT VALIDATION
# Asserts required columns, uniqueness, dtype normalization, label validity.
# -------------------------------------------------------------------
required_union_cols = {
    "union_id", "dataset_id", "well_id", "crop_id",
    "union_centroid_well_y_px", "union_centroid_well_x_px",
    "union_medoid_well_y_px", "union_medoid_well_x_px",
}
required_mega_cols = {
    "union_id", "dataset_id", "well_id", "crop_id",
}
required_ann_cols = {
    "annotation_id", "dataset_id", "well_id", "crop_id",
    "label", "confidence",
    "raw_click_well_y_px", "raw_click_well_x_px",
}
required_crop_cols = {
    "crop_id", "dataset_id", "well_id",
    "well_ymin_px", "well_xmin_px", "well_ymax_px", "well_xmax_px",
    "annotator_status", "crop_registry_version",
}

missing_union = sorted(required_union_cols - set(candidate_union.columns))
missing_mega  = sorted(required_mega_cols  - set(mega_feature_table.columns))
missing_ann   = sorted(required_ann_cols   - set(annotations.columns))
missing_crop  = sorted(required_crop_cols  - set(crop_registry.columns))

assert not missing_union, f"candidate_union missing columns: {missing_union}"
assert not missing_mega,  f"mega_feature_table missing columns: {missing_mega}"
assert not missing_ann,   f"annotations missing columns: {missing_ann}"
assert not missing_crop,  f"crop_registry missing columns: {missing_crop}"

assert candidate_union["union_id"].is_unique,       "candidate_union: union_id must be unique"
assert mega_feature_table["union_id"].is_unique,    "mega_feature_table: union_id must be unique"
assert annotations["annotation_id"].is_unique,      "annotations: annotation_id must be unique"
assert crop_registry["crop_id"].is_unique,           "crop_registry: crop_id must be unique"

# Normalize ID column dtypes to str (guard against int crop_id from parquet vs str from yaml)
for df in [candidate_union, mega_feature_table, annotations, crop_registry]:
    for c in ["dataset_id", "well_id", "crop_id"]:
        if c in df.columns:
            df[c] = df[c].astype(str)

for c in ["well_ymin_px", "well_xmin_px", "well_ymax_px", "well_xmax_px"]:
    crop_registry[c] = crop_registry[c].astype(float)

valid_labels = {"TRUE_SPOT", "UNCERTAIN"}
actual_labels = set(annotations["label"].astype(str).unique())
unexpected_labels = actual_labels - valid_labels
if unexpected_labels:
    log(f"[warn] Unexpected annotation labels (will be treated as excluded): {unexpected_labels}")

log("Contract validation passed.")

In [ ]:
# -------------------------------------------------------------------
# ANNOTATION COORDINATE PREPARATION
# Use refined_well coords; fall back to raw_well; record source per row.
# -------------------------------------------------------------------
def choose_truth_well_coords(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add truth_well_y/x_px and truth_coord_source columns.
    Prefers refined_click_well_y/x_px; falls back to raw_click_well_y/x_px.
    Also reconstructs well coords from crop coords + origin if neither is present.
    """
    out = df.copy()

    def _to_float(series):
        return pd.to_numeric(series, errors="coerce") if series.name in out.columns else pd.Series(np.nan, index=out.index)

    ry = _to_float(out.get("refined_click_well_y_px", pd.Series(np.nan, index=out.index, name="refined_click_well_y_px")))
    rx = _to_float(out.get("refined_click_well_x_px", pd.Series(np.nan, index=out.index, name="refined_click_well_x_px")))
    wy = _to_float(out.get("raw_click_well_y_px",     pd.Series(np.nan, index=out.index, name="raw_click_well_y_px")))
    wx = _to_float(out.get("raw_click_well_x_px",     pd.Series(np.nan, index=out.index, name="raw_click_well_x_px")))

    out["truth_well_y_px"]    = np.where(np.isfinite(ry), ry, wy)
    out["truth_well_x_px"]    = np.where(np.isfinite(rx), rx, wx)
    out["truth_coord_source"] = np.where(
        np.isfinite(ry) & np.isfinite(rx), "refined_well", "raw_well"
    )
    return out

annotations = choose_truth_well_coords(annotations)

bad_truth_rows = annotations[
    ~np.isfinite(annotations["truth_well_y_px"]) |
    ~np.isfinite(annotations["truth_well_x_px"])
]
if len(bad_truth_rows) > 0:
    log(f"[warn] {len(bad_truth_rows)} annotation rows with invalid truth coordinates — these will be skipped during matching.")
    annotations = annotations[
        np.isfinite(annotations["truth_well_y_px"]) &
        np.isfinite(annotations["truth_well_x_px"])
    ].reset_index(drop=True)

n_true    = int((annotations["label"] == "TRUE_SPOT").sum())
n_unc     = int((annotations["label"] == "UNCERTAIN").sum())
log(f"Annotations ready: {len(annotations):,} total — {n_true:,} TRUE_SPOT, {n_unc:,} UNCERTAIN")
log(f"Using refined coords: {(annotations['truth_coord_source'] == 'refined_well').sum():,} | raw fallback: {(annotations['truth_coord_source'] == 'raw_well').sum():,}")
display(annotations[["annotation_id", "well_id", "crop_id", "label", "truth_well_y_px", "truth_well_x_px", "truth_coord_source"]].head())

In [ ]:
# -------------------------------------------------------------------
# CROP TERRITORY / COMPLETION RESOLUTION
# Determines which crops are eligible for negative generation.
# -------------------------------------------------------------------
ann_counts = (
    annotations
    .groupby(["dataset_id", "well_id", "crop_id", "label"], dropna=False)
    .size().rename("n_annotations").reset_index()
)
ann_counts_wide = ann_counts.pivot_table(
    index=["dataset_id", "well_id", "crop_id"],
    columns="label",
    values="n_annotations",
    aggfunc="sum",
    fill_value=0,
).reset_index()
for col in ["TRUE_SPOT", "UNCERTAIN"]:
    if col not in ann_counts_wide.columns:
        ann_counts_wide[col] = 0

crop_status = crop_registry.merge(ann_counts_wide, on=["dataset_id", "well_id", "crop_id"], how="left")
crop_status["TRUE_SPOT"]  = crop_status["TRUE_SPOT"].fillna(0).astype(int)
crop_status["UNCERTAIN"]  = crop_status["UNCERTAIN"].fillna(0).astype(int)
crop_status["annotator_status_norm"] = crop_status["annotator_status"].astype(str).str.strip().str.lower()

done_like = {str(x).strip().lower() for x in CFG["done_like_crop_statuses"]}
crop_status["is_done_like"] = crop_status["annotator_status_norm"].isin(done_like)

if CFG["require_annotations_present_for_negative_generation"]:
    crop_status["is_completed_for_negatives"] = (
        crop_status["is_done_like"] &
        ((crop_status["TRUE_SPOT"] + crop_status["UNCERTAIN"]) > 0)
    )
else:
    crop_status["is_completed_for_negatives"] = crop_status["is_done_like"]

# Merge completion flag onto candidate_union
candidate_union = candidate_union.merge(
    crop_status[[
        "dataset_id", "well_id", "crop_id",
        "annotator_status", "crop_registry_version",
        "is_completed_for_negatives", "TRUE_SPOT", "UNCERTAIN"
    ]],
    on=["dataset_id", "well_id", "crop_id"],
    how="left",
    suffixes=("", "_crop"),
)

# Merge onto mega: resolve eligible_for_negative_supervision per candidate
mega_feature_table = mega_feature_table.merge(
    crop_status[["dataset_id", "well_id", "crop_id", "is_completed_for_negatives"]],
    on=["dataset_id", "well_id", "crop_id"],
    how="left",
    suffixes=("", "_crop"),
)
if (
    CFG["use_mega_annotation_coverage_if_available"]
    and "annotation_coverage_status" in mega_feature_table.columns
):
    ok_cov = set(CFG["accepted_annotation_coverage_statuses_for_negatives"])
    mega_feature_table["coverage_allows_negative"] = (
        mega_feature_table["annotation_coverage_status"].astype(str).isin(ok_cov)
    )
    mega_feature_table["eligible_for_negative_supervision"] = (
        mega_feature_table["coverage_allows_negative"]
        & mega_feature_table["is_completed_for_negatives"].fillna(False)
    )
else:
    mega_feature_table["eligible_for_negative_supervision"] = (
        mega_feature_table["is_completed_for_negatives"].fillna(False)
    )

# Pull eligible_for_negative_supervision back onto candidate_union
candidate_union = candidate_union.merge(
    mega_feature_table[["union_id", "eligible_for_negative_supervision"]].drop_duplicates("union_id"),
    on="union_id",
    how="left",
)

FULLY_ANNOTATED_CROPS = set(
    crop_status.loc[crop_status["is_completed_for_negatives"], "crop_id"].unique()
)
log(f"Crops eligible for negative generation: {len(FULLY_ANNOTATED_CROPS)}")

annotated_crop_ids  = set(annotations["crop_id"].dropna().unique())
candidate_crop_ids  = set(candidate_union["crop_id"].dropna().unique())
overlap_crops       = annotated_crop_ids & candidate_crop_ids
log(f"Annotated crops: {len(annotated_crop_ids)} | Candidate crops: {len(candidate_crop_ids)} | Overlap: {len(overlap_crops)}")

assert len(overlap_crops) > 0, (
    "No overlap between annotated crops and candidate crops. "
    "Check crop_id consistency between NB02 and NB03."
)

display(
    crop_status[[
        "dataset_id", "well_id", "crop_id", "annotator_status",
        "is_completed_for_negatives", "TRUE_SPOT", "UNCERTAIN"
    ]].sort_values(["well_id", "crop_id"])
)

In [ ]:
# -------------------------------------------------------------------
# MATCHING HELPERS
# -------------------------------------------------------------------
def stable_match_id(union_id, annotation_id, match_status, radius, registry_version) -> str:
    raw = f"{union_id}|{annotation_id}|{match_status}|{radius}|{registry_version}"
    return "match_" + hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]

def euclidean_distance_matrix(cand_yx: np.ndarray, truth_yx: np.ndarray) -> np.ndarray:
    if len(cand_yx) == 0 or len(truth_yx) == 0:
        return np.empty((len(cand_yx), len(truth_yx)), dtype=np.float32)
    dy = cand_yx[:, None, 0] - truth_yx[None, :, 0]
    dx = cand_yx[:, None, 1] - truth_yx[None, :, 1]
    return np.sqrt(dy * dy + dx * dx).astype(np.float32)

def thresholded_hungarian(cand_yx: np.ndarray, truth_yx: np.ndarray, radius: float):
    """Returns list of (cand_idx, truth_idx, distance_px) for matched pairs."""
    D = euclidean_distance_matrix(cand_yx, truth_yx)
    if D.size == 0:
        return [], D
    BIG = 1e9
    C = D.copy()
    C[C > radius] = BIG
    rows, cols = linear_sum_assignment(C)
    kept = [(int(r), int(c), float(D[r, c])) for r, c in zip(rows, cols) if C[r, c] < BIG]
    return kept, D

def halo_exclude_candidates(cand_yx: np.ndarray, uncertain_yx: np.ndarray, radius: float) -> set:
    """
    Returns the set of candidate indices that fall within `radius` px
    of ANY uncertain annotation (halo exclusion policy).
    """
    if len(cand_yx) == 0 or len(uncertain_yx) == 0:
        return set()
    tree = cKDTree(uncertain_yx)
    excluded = set()
    for i, cy in enumerate(cand_yx):
        idxs = tree.query_ball_point(cy, r=radius)
        if idxs:
            excluded.add(i)
    return excluded

def nearest_truth_distance(cand_yx: np.ndarray, truth_yx: np.ndarray) -> np.ndarray:
    if len(cand_yx) == 0:
        return np.array([], dtype=np.float32)
    if len(truth_yx) == 0:
        return np.full(len(cand_yx), np.inf, dtype=np.float32)
    tree = cKDTree(truth_yx)
    dist, _ = tree.query(cand_yx, k=1)
    return np.asarray(dist, dtype=np.float32)

In [ ]:
# -------------------------------------------------------------------
# PER-CROP HUNGARIAN MATCHING  (§9.1)
#
# Per crop:
#   1. Resolve TRUE_SPOT and UNCERTAIN annotations
#   2. Halo-exclude candidates near UNCERTAIN (before Hungarian)
#   3. Run thresholded Hungarian on remaining candidates vs TRUE_SPOT
#   4. Classify each candidate and truth point:
#      matched_positive | unmatched_candidate_negative |
#      unmatched_truth_false_negative | excluded_uncertain
# -------------------------------------------------------------------
log("=" * 90)
log("BEGIN MATCHING")
t0_match = time.perf_counter()

MATCH_RADIUS  = CFG["truth_match_radius_px"]
HALO_RADIUS   = CFG["uncertain_halo_radius_px"]
MATCH_VERSION = CFG["matching_registry_version"]
EXCL_POLICY   = CFG["uncertain_exclusion_policy"]

MATCH_ROWS         = []
TRUTH_SUMMARY_ROWS = []
MATCH_TIMINGS      = []
CROP_SUMMARY_ROWS  = []

# Pre-index by (dataset_id, well_id, crop_id) for O(1) lookup
crop_union_groups = {k: g for k, g in candidate_union.groupby(["dataset_id", "well_id", "crop_id"], sort=True)}
crop_ann_groups   = {k: g for k, g in annotations.groupby(["dataset_id", "well_id", "crop_id"], sort=True)}

all_crop_keys = sorted(set(crop_union_groups.keys()) | set(crop_ann_groups.keys()))
n_keys = len(all_crop_keys)

for idx, crop_key in enumerate(all_crop_keys, 1):
    kt0 = time.perf_counter()
    dataset_id, well_id, crop_id = crop_key

    cand = crop_union_groups.get(crop_key, pd.DataFrame()).copy()
    ann  = crop_ann_groups.get(crop_key, pd.DataFrame()).copy()

    if len(cand) == 0 and len(ann) == 0:
        continue

    if len(cand) > 0:
        cand = cand.sort_values("union_id").reset_index(drop=True)
    if len(ann) > 0:
        ann  = ann.sort_values("annotation_id").reset_index(drop=True)

    n_true = int((ann["label"] == "TRUE_SPOT").sum()) if len(ann) else 0
    n_unc  = int((ann["label"] == "UNCERTAIN").sum()) if len(ann) else 0
    log(f"[{idx}/{n_keys}] {well_id}/{crop_id[-12:]}  candidates={len(cand)}  true={n_true}  uncertain={n_unc}")

    true_ann     = ann[ann["label"] == "TRUE_SPOT"].reset_index(drop=True)
    uncertain_ann = ann[ann["label"] == "UNCERTAIN"].reset_index(drop=True)

    cand_yx = cand[["union_medoid_well_y_px", "union_medoid_well_x_px"]].to_numpy(dtype=np.float32)
    true_yx = (true_ann[["truth_well_y_px", "truth_well_x_px"]].to_numpy(dtype=np.float32)
               if len(true_ann) else np.empty((0, 2), dtype=np.float32))
    unc_yx  = (uncertain_ann[["truth_well_y_px", "truth_well_x_px"]].to_numpy(dtype=np.float32)
               if len(uncertain_ann) else np.empty((0, 2), dtype=np.float32))

    # ── Step 1: UNCERTAIN halo exclusion ──────────────────────────────────
    if EXCL_POLICY == "halo_exclusion":
        excluded_cand_idx = halo_exclude_candidates(cand_yx, unc_yx, HALO_RADIUS)
    else:
        # assigned_only: will handle below during nearest-neighbor pass
        excluded_cand_idx = set()

    # Non-excluded candidates enter Hungarian
    eligible_mask = np.array([i not in excluded_cand_idx for i in range(len(cand))])
    cand_eligible = cand[eligible_mask].reset_index(drop=True)
    cand_yx_elig  = cand_yx[eligible_mask]
    elig_indices  = np.where(eligible_mask)[0]  # maps back to cand row index

    # ── Step 2: Hungarian on TRUE_SPOT ────────────────────────────────────
    kept, D_true = thresholded_hungarian(cand_yx_elig, true_yx, MATCH_RADIUS)

    matched_cand_idx_orig = set()  # indices into cand (original)
    matched_true_idx      = set()

    for r, c, d in kept:
        orig_r = int(elig_indices[r])
        matched_cand_idx_orig.add(orig_r)
        matched_true_idx.add(c)

        crow = cand.iloc[orig_r]
        arow = true_ann.iloc[c]

        MATCH_ROWS.append({
            "match_id":                 stable_match_id(crow["union_id"], arow["annotation_id"], "matched_positive", MATCH_RADIUS, MATCH_VERSION),
            "union_id":                 crow["union_id"],
            "annotation_id":            arow["annotation_id"],
            "dataset_id":               dataset_id,
            "well_id":                  well_id,
            "crop_id":                  crop_id,
            "match_status":             "matched_positive",
            "gt_label":                 "TRUE_SPOT",
            "gt_distance_px":           float(d),
            "gt_click_well_y_px":       float(arow["truth_well_y_px"]),
            "gt_click_well_x_px":       float(arow["truth_well_x_px"]),
            "candidate_well_y_px":      float(crow["union_medoid_well_y_px"]),
            "candidate_well_x_px":      float(crow["union_medoid_well_x_px"]),
            "annotation_confidence":    arow.get("confidence", np.nan),
            "annotation_label_original": arow.get("label", None),
            "truth_coord_source":       arow.get("truth_coord_source", None),
            "matching_algorithm":       CFG["matching_algorithm"],
            "truth_match_radius_px":    float(MATCH_RADIUS),
            "matching_registry_version": MATCH_VERSION,
        })

    # ── Step 3: excluded_uncertain rows ───────────────────────────────────
    # Halo policy: record one row per excluded candidate (nearest uncertain)
    for ei in excluded_cand_idx:
        crow = cand.iloc[ei]
        cy   = cand_yx[ei]
        if len(unc_yx):
            dists  = np.sqrt(np.sum((unc_yx - cy) ** 2, axis=1))
            nn_idx = int(np.argmin(dists))
            arow   = uncertain_ann.iloc[nn_idx]
            ann_id = arow["annotation_id"]
            gt_y   = float(arow["truth_well_y_px"])
            gt_x   = float(arow["truth_well_x_px"])
            dist_v = float(dists[nn_idx])
        else:
            ann_id = None; gt_y = np.nan; gt_x = np.nan; dist_v = np.nan

        MATCH_ROWS.append({
            "match_id":                 stable_match_id(crow["union_id"], ann_id, "excluded_uncertain", HALO_RADIUS, MATCH_VERSION),
            "union_id":                 crow["union_id"],
            "annotation_id":            ann_id,
            "dataset_id":               dataset_id,
            "well_id":                  well_id,
            "crop_id":                  crop_id,
            "match_status":             "excluded_uncertain",
            "gt_label":                 "UNCERTAIN",
            "gt_distance_px":           dist_v,
            "gt_click_well_y_px":       gt_y,
            "gt_click_well_x_px":       gt_x,
            "candidate_well_y_px":      float(cand_yx[ei][0]),
            "candidate_well_x_px":      float(cand_yx[ei][1]),
            "annotation_confidence":    np.nan,
            "annotation_label_original": "UNCERTAIN",
            "truth_coord_source":       None,
            "matching_algorithm":       CFG["matching_algorithm"],
            "truth_match_radius_px":    float(MATCH_RADIUS),
            "matching_registry_version": MATCH_VERSION,
        })

    # assigned_only fallback: nearest-candidate per uncertain
    if EXCL_POLICY == "assigned_only" and len(uncertain_ann) > 0 and len(cand) > 0:
        for uidx, urow in uncertain_ann.iterrows():
            dists = np.sqrt(np.sum((cand_yx - np.array([urow["truth_well_y_px"], urow["truth_well_x_px"]])) ** 2, axis=1))
            nn = int(np.argmin(dists))
            if dists[nn] <= MATCH_RADIUS:
                uid_nn = cand.iloc[nn]["union_id"]
                if nn not in matched_cand_idx_orig and nn not in excluded_cand_idx:
                    excluded_cand_idx.add(nn)
                    crow = cand.iloc[nn]
                    MATCH_ROWS.append({
                        "match_id":                 stable_match_id(uid_nn, urow["annotation_id"], "excluded_uncertain", MATCH_RADIUS, MATCH_VERSION),
                        "union_id":                 uid_nn,
                        "annotation_id":            urow["annotation_id"],
                        "dataset_id":               dataset_id,
                        "well_id":                  well_id,
                        "crop_id":                  crop_id,
                        "match_status":             "excluded_uncertain",
                        "gt_label":                 "UNCERTAIN",
                        "gt_distance_px":           float(dists[nn]),
                        "gt_click_well_y_px":       float(urow["truth_well_y_px"]),
                        "gt_click_well_x_px":       float(urow["truth_well_x_px"]),
                        "candidate_well_y_px":      float(crow["union_medoid_well_y_px"]),
                        "candidate_well_x_px":      float(crow["union_medoid_well_x_px"]),
                        "annotation_confidence":    np.nan,
                        "annotation_label_original": "UNCERTAIN",
                        "truth_coord_source":       None,
                        "matching_algorithm":       CFG["matching_algorithm"],
                        "truth_match_radius_px":    float(MATCH_RADIUS),
                        "matching_registry_version": MATCH_VERSION,
                    })

    # ── Step 4: unmatched candidates → negative (completed crops only) ─────
    is_neg_eligible = bool(cand.iloc[0].get("eligible_for_negative_supervision", False)) if len(cand) else False
    if crop_id in FULLY_ANNOTATED_CROPS and is_neg_eligible:
        for ci in range(len(cand)):
            if ci in matched_cand_idx_orig or ci in excluded_cand_idx:
                continue
            crow = cand.iloc[ci]
            MATCH_ROWS.append({
                "match_id":                 stable_match_id(crow["union_id"], "NONE", "unmatched_candidate_negative", MATCH_RADIUS, MATCH_VERSION),
                "union_id":                 crow["union_id"],
                "annotation_id":            None,
                "dataset_id":               dataset_id,
                "well_id":                  well_id,
                "crop_id":                  crop_id,
                "match_status":             "unmatched_candidate_negative",
                "gt_label":                 "NEGATIVE",
                "gt_distance_px":           np.nan,
                "gt_click_well_y_px":       np.nan,
                "gt_click_well_x_px":       np.nan,
                "candidate_well_y_px":      float(cand_yx[ci][0]),
                "candidate_well_x_px":      float(cand_yx[ci][1]),
                "annotation_confidence":    np.nan,
                "annotation_label_original": None,
                "truth_coord_source":       None,
                "matching_algorithm":       CFG["matching_algorithm"],
                "truth_match_radius_px":    float(MATCH_RADIUS),
                "matching_registry_version": MATCH_VERSION,
            })

    # ── Step 5: unmatched truth → false negative (diagnostic only) ─────────
    for ai in range(len(true_ann)):
        if ai in matched_true_idx:
            continue
        arow = true_ann.iloc[ai]
        nearest_cand_dist = (
            float(np.min(D_true[:, ai])) if D_true.size > 0 and D_true.shape[1] > ai
            else np.nan
        )
        MATCH_ROWS.append({
            "match_id":                 stable_match_id("NONE", arow["annotation_id"], "unmatched_truth_false_negative", MATCH_RADIUS, MATCH_VERSION),
            "union_id":                 None,
            "annotation_id":            arow["annotation_id"],
            "dataset_id":               dataset_id,
            "well_id":                  well_id,
            "crop_id":                  crop_id,
            "match_status":             "unmatched_truth_false_negative",
            "gt_label":                 "TRUE_SPOT",
            "gt_distance_px":           nearest_cand_dist,
            "gt_click_well_y_px":       float(arow["truth_well_y_px"]),
            "gt_click_well_x_px":       float(arow["truth_well_x_px"]),
            "candidate_well_y_px":      np.nan,
            "candidate_well_x_px":      np.nan,
            "annotation_confidence":    arow.get("confidence", np.nan),
            "annotation_label_original": "TRUE_SPOT",
            "truth_coord_source":       arow.get("truth_coord_source", None),
            "matching_algorithm":       CFG["matching_algorithm"],
            "truth_match_radius_px":    float(MATCH_RADIUS),
            "matching_registry_version": MATCH_VERSION,
        })
        TRUTH_SUMMARY_ROWS.append({
            "annotation_id":             arow["annotation_id"],
            "dataset_id":                dataset_id,
            "well_id":                   well_id,
            "crop_id":                   crop_id,
            "label":                     "TRUE_SPOT",
            "truth_well_y_px":           float(arow["truth_well_y_px"]),
            "truth_well_x_px":           float(arow["truth_well_x_px"]),
            "match_status":              "unmatched_truth_false_negative",
            "matched_union_id":          None,
            "nearest_union_distance_px": nearest_cand_dist,
            "matching_registry_version": MATCH_VERSION,
        })

    MATCH_TIMINGS.append({"crop_id": crop_id, "well_id": well_id, "timing_sec": time.perf_counter() - kt0})
    CROP_SUMMARY_ROWS.append({
        "crop_id": crop_id, "well_id": well_id,
        "n_candidates":         len(cand),
        "n_truth":              n_true,
        "n_uncertain":          n_unc,
        "n_matched":            len(matched_cand_idx_orig),
        "n_negative":           sum(1 for ci in range(len(cand)) if ci not in matched_cand_idx_orig and ci not in excluded_cand_idx)
                                if crop_id in FULLY_ANNOTATED_CROPS else 0,
        "n_excluded_uncertain": len(excluded_cand_idx),
        "n_false_negative":     n_true - len(matched_true_idx),
        "is_fully_annotated":   crop_id in FULLY_ANNOTATED_CROPS,
    })

log(f"Matching done in {time.perf_counter() - t0_match:.1f}s")

candidate_to_truth_match = pd.DataFrame(MATCH_ROWS)
truth_evaluation_summary = pd.DataFrame(TRUTH_SUMMARY_ROWS)
matching_timing_df       = pd.DataFrame(MATCH_TIMINGS)
crop_match_summary       = pd.DataFrame(CROP_SUMMARY_ROWS)

if len(candidate_to_truth_match) == 0:
    raise RuntimeError("No match rows produced. Check crop registry, annotations, and candidate inputs.")

candidate_to_truth_match = candidate_to_truth_match.sort_values(
    ["dataset_id", "well_id", "crop_id", "match_status", "union_id", "annotation_id"],
    na_position="last"
).reset_index(drop=True)

display(candidate_to_truth_match["match_status"].value_counts(dropna=False).rename_axis("match_status").to_frame("n"))

In [ ]:
# -------------------------------------------------------------------
# MATCH TABLE VALIDATION  (§3.7)
# -------------------------------------------------------------------
required_match_cols = {
    "match_id", "union_id", "annotation_id", "match_status", "gt_label",
    "gt_distance_px", "matching_algorithm", "truth_match_radius_px",
    "matching_registry_version",
}
missing_match_cols = sorted(required_match_cols - set(candidate_to_truth_match.columns))
assert not missing_match_cols, f"candidate_to_truth_match missing columns: {missing_match_cols}"

allowed_match_status = {
    "matched_positive",
    "unmatched_candidate_negative",
    "unmatched_truth_false_negative",
    "excluded_uncertain",
}
bad_match_status = sorted(set(candidate_to_truth_match["match_status"]) - allowed_match_status)
assert not bad_match_status, f"Invalid match_status values: {bad_match_status}"
assert candidate_to_truth_match["match_id"].is_unique, "match_id must be unique"

# One-to-one for positives
_pos = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "matched_positive"]
assert _pos["union_id"].is_unique,      "union_id appears in multiple matched_positive rows"
assert _pos["annotation_id"].is_unique, "annotation_id appears in multiple matched_positive rows"

# Matched positives must have both IDs
bad_pos = _pos[_pos["union_id"].isna() | _pos["annotation_id"].isna()]
assert len(bad_pos) == 0, "matched_positive rows must have both union_id and annotation_id"

n_pos  = int((candidate_to_truth_match["match_status"] == "matched_positive").sum())
n_neg  = int((candidate_to_truth_match["match_status"] == "unmatched_candidate_negative").sum())
n_fn   = int((candidate_to_truth_match["match_status"] == "unmatched_truth_false_negative").sum())
n_excl = int((candidate_to_truth_match["match_status"] == "excluded_uncertain").sum())

log(f"match_status counts: pos={n_pos}  neg={n_neg}  fn={n_fn}  excl={n_excl}")
if n_pos + n_fn > 0:
    log(f"Detection recall (pre-classifier): {n_pos / (n_pos + n_fn):.3f}  ({n_pos}/{n_pos + n_fn})")
if n_pos + n_neg > 0:
    log(f"Positive rate in supervision:      {n_pos / (n_pos + n_neg):.4f}  ({n_pos}/{n_pos + n_neg})")
log("candidate_to_truth_match validation passed.")

In [ ]:
# -------------------------------------------------------------------
# MATCH DIAGNOSTICS
# -------------------------------------------------------------------
log("=" * 90)
log("MATCH DIAGNOSTICS")

matched   = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "matched_positive"]
false_neg = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "unmatched_truth_false_negative"]

if len(matched) > 0:
    dists = matched["gt_distance_px"].dropna()
    log(f"\nMatched pair distances (px): mean={dists.mean():.2f}  median={dists.median():.2f}  "
        f"std={dists.std():.2f}  p90={dists.quantile(0.9):.2f}  max={dists.max():.2f}")

if len(false_neg) > 0:
    fn_dists = false_neg["gt_distance_px"].dropna()
    log(f"\nFalse negatives ({len(false_neg)}): nearest cand dist: "
        f"mean={fn_dists.mean():.2f}  median={fn_dists.median():.2f}  "
        f"min={fn_dists.min():.2f}  max={fn_dists.max():.2f}")
    n_near = int((fn_dists < MATCH_RADIUS * 2).sum())
    log(f"  Within 2× radius ({MATCH_RADIUS*2:.0f}px): {n_near} near-misses | {len(fn_dists) - n_near} detection failures")

log(f"\nRecall sensitivity to matching radius:")
total_truth = n_pos + n_fn
for test_r in [3.0, 5.0, 8.0, 10.0, 12.0, 15.0]:
    n_at_r = int((matched["gt_distance_px"] <= test_r).sum()) if len(matched) > 0 else 0
    log(f"  r={test_r:.0f}px  recall={n_at_r / max(total_truth, 1):.3f}  ({n_at_r}/{total_truth})")

# Per-crop summary table
match_counts_pivot = (
    candidate_to_truth_match
    .groupby(["dataset_id", "well_id", "crop_id", "match_status"], dropna=False)
    .size().rename("n_rows").reset_index()
    .pivot_table(index=["dataset_id", "well_id", "crop_id"], columns="match_status", values="n_rows", aggfunc="sum", fill_value=0)
    .reset_index()
)
for col in ["matched_positive", "unmatched_candidate_negative", "unmatched_truth_false_negative", "excluded_uncertain"]:
    if col not in match_counts_pivot.columns:
        match_counts_pivot[col] = 0
match_counts_pivot["precision_proxy"] = (
    match_counts_pivot["matched_positive"] /
    (match_counts_pivot["matched_positive"] + match_counts_pivot["unmatched_candidate_negative"] + 1e-9)
)
match_counts_pivot["recall_proxy"] = (
    match_counts_pivot["matched_positive"] /
    (match_counts_pivot["matched_positive"] + match_counts_pivot["unmatched_truth_false_negative"] + 1e-9)
)
display(match_counts_pivot.sort_values(["well_id", "crop_id"]))

# Diagnostic plots
if CFG["write_match_diagnostics"] and len(matched) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(matched["gt_distance_px"].dropna(), bins=30, edgecolor="black", linewidth=0.5)
    axes[0].axvline(MATCH_RADIUS, color="red", linestyle="--", label=f"radius={MATCH_RADIUS}px")
    axes[0].set_xlabel("match distance (px)"); axes[0].set_ylabel("count")
    axes[0].set_title("Matched pair distances"); axes[0].legend()

    if len(crop_match_summary) > 0:
        x = range(len(crop_match_summary))
        axes[1].bar(x, crop_match_summary["n_matched"], label="matched", alpha=0.8)
        axes[1].bar(x, crop_match_summary["n_false_negative"],
                    bottom=crop_match_summary["n_matched"], label="false neg", alpha=0.8)
        axes[1].set_xlabel("crop index"); axes[1].set_ylabel("count")
        axes[1].set_title("Per-crop: matched + FN"); axes[1].legend()

    if len(false_neg) > 0:
        axes[2].hist(false_neg["gt_distance_px"].dropna(), bins=30, edgecolor="black", linewidth=0.5)
        axes[2].axvline(MATCH_RADIUS, color="red", linestyle="--", label=f"radius={MATCH_RADIUS}px")
        axes[2].set_xlabel("nearest candidate distance (px)")
        axes[2].set_title("FN: dist to nearest candidate"); axes[2].legend()

    fig.tight_layout()
    diag_path = REPORT_DIR / f"{RUN_ID}_match_diagnostics.png"
    fig.savefig(diag_path, dpi=CFG["qa_fig_dpi"], bbox_inches="tight")
    plt.close(fig)
    log(f"Diagnostics plot saved: {diag_path.name}")

In [ ]:
# -------------------------------------------------------------------
# SPATIAL BLOCK SPLITS  (§3.10, §10.2)
#
# Each annotated crop = one spatial block.
# Deterministic hash-based assignment, stratified by well_id.
# Reuses existing splits.parquet if registry version matches.
# -------------------------------------------------------------------
log("=" * 90)
log("SPATIAL BLOCK SPLITS")

SPLIT_REGISTRY_VERSION = CFG["split_registry_version"]
existing_splits_path   = latest_matching_file(TABLES_DIR, ["*splits.parquet"])
splits_df              = None

if CFG["reuse_existing_splits"] and existing_splits_path is not None:
    _existing = pd.read_parquet(existing_splits_path)
    if (
        "split_registry_version" in _existing.columns
        and (_existing["split_registry_version"] == SPLIT_REGISTRY_VERSION).all()
        and set(_existing["crop_id"]).issuperset(FULLY_ANNOTATED_CROPS)
    ):
        splits_df = _existing
        log(f"Reusing existing splits.parquet ({len(splits_df)} rows) from {existing_splits_path.name}")
    else:
        log(f"Existing splits.parquet version mismatch or incomplete — regenerating.")

if splits_df is None:
    TRAIN_FRAC = CFG["split_train_frac"]
    CAL_FRAC   = CFG["split_cal_frac"]
    SPLIT_SEED = CFG["split_seed"]

    def deterministic_split_role(crop_id: str, well_id: str, seed: int) -> str:
        h   = hashlib.sha256(f"{crop_id}|{well_id}|{seed}".encode()).hexdigest()
        val = int(h[:8], 16) / 0xFFFFFFFF
        if val < TRAIN_FRAC:
            return "train"
        elif val < TRAIN_FRAC + CAL_FRAC:
            return "calibration"
        return "test"

    crop_reg_idx = crop_registry.set_index("crop_id")
    split_rows = []
    for cid in sorted(FULLY_ANNOTATED_CROPS):
        if cid not in crop_reg_idx.index:
            log(f"[warn] crop_id {cid} not found in crop_registry — skipping split assignment")
            continue
        cr   = crop_reg_idx.loc[cid]
        role = deterministic_split_role(cid, cr["well_id"], SPLIT_SEED)
        split_rows.append({
            "split_id":              sha1_text(f"{cid}|{role}|{SPLIT_REGISTRY_VERSION}"),
            "dataset_id":            cr["dataset_id"],
            "well_id":               cr["well_id"],
            "crop_id":               cid,
            "spatial_block_id":      cid,
            "split_role":            role,
            "split_registry_version": SPLIT_REGISTRY_VERSION,
        })
    splits_df = pd.DataFrame(split_rows)
    log(f"Generated {len(splits_df)} split assignments.")

assert splits_df["crop_id"].is_unique, "Split leakage: crop_id appears in multiple splits!"
assert set(splits_df["split_role"]).issubset({"train", "calibration", "test"})

role_counts = splits_df["split_role"].value_counts()
log(f"\nSplit assignment ({len(splits_df)} crops):")
for role in ["train", "calibration", "test"]:
    log(f"  {role}: {role_counts.get(role, 0)} crops")
log("\nPer-well split distribution:")
display(splits_df.groupby(["well_id", "split_role"]).size().unstack(fill_value=0))
log("No split leakage ✓")

In [ ]:
# -------------------------------------------------------------------
# BUILD training_supervision_table  (§3.9)
#
# Left-join mega_feature_table + match results + splits.
# Every union_id gets exactly one row.
# supervision_status ∈ {included, excluded}
# target_binary ∈ {0, 1, NaN}  (NaN for excluded rows)
# -------------------------------------------------------------------
log("=" * 90)
log("BUILDING training_supervision_table")

# ── Supervision mapping ────────────────────────────────────────────────────
def supervision_mapping(row) -> tuple:
    ms = row["match_status"]
    if ms == "matched_positive":
        return "included", 1.0, "matched_true_spot"
    elif ms == "unmatched_candidate_negative":
        return "included", 0.0, "annotated_territory_unmatched_candidate"
    elif ms == "excluded_uncertain":
        return "excluded", np.nan, "uncertain_annotation_exclusion"
    else:
        return "excluded", np.nan, f"unsupported_{ms}"

union_backed = candidate_to_truth_match[candidate_to_truth_match["union_id"].notna()].copy()
mapped       = union_backed.apply(supervision_mapping, axis=1, result_type="expand")
mapped.columns = ["supervision_status", "target_binary", "target_source"]
union_backed = pd.concat([union_backed.reset_index(drop=True), mapped], axis=1)

# ── Sample weight computation ──────────────────────────────────────────────
supervised_only = union_backed[union_backed["target_binary"].notna()]
n_pos_w = int(supervised_only["target_binary"].sum())
n_neg_w = int(len(supervised_only) - n_pos_w)
n_total_w = n_pos_w + n_neg_w

if CFG["positive_sample_weight_override"] is not None:
    w_pos = float(CFG["positive_sample_weight_override"])
    w_neg = float(CFG["negative_sample_weight_override"] or 1.0)
    log(f"Sample weights (override): w_pos={w_pos:.4f}, w_neg={w_neg:.4f}")
else:
    strategy = CFG["sample_weight_strategy"]
    if strategy == "balanced":
        w_pos = n_total_w / (2 * max(n_pos_w, 1))
        w_neg = n_total_w / (2 * max(n_neg_w, 1))
    elif strategy == "sqrt_inverse_freq":
        freq_pos = n_pos_w / max(n_total_w, 1)
        freq_neg = n_neg_w / max(n_total_w, 1)
        w_pos = 1.0 / max(math.sqrt(freq_pos), 1e-6)
        w_neg = 1.0 / max(math.sqrt(freq_neg), 1e-6)
        w_mean = (w_pos * n_pos_w + w_neg * n_neg_w) / max(n_total_w, 1)
        w_pos /= max(w_mean, 1e-6)
        w_neg /= max(w_mean, 1e-6)
    else:  # "none"
        w_pos = 1.0
        w_neg = 1.0
    log(f"Sample weights ({strategy}): w_pos={w_pos:.4f}, w_neg={w_neg:.4f}")

union_backed["sample_weight"] = (
    union_backed["target_binary"]
    .map({1.0: w_pos, 0.0: w_neg})
    .fillna(float(CFG["excluded_sample_weight"]))
)

# ── Join splits ────────────────────────────────────────────────────────────
union_backed = union_backed.merge(
    splits_df[["crop_id", "split_id", "split_role"]],
    on="crop_id", how="left"
)

# ── Left-join mega_feature_table (primary) + supervision results ───────────
# Exclude heavy provenance cols from mega to keep the supervision table clean
_meta_exclude = {"run_id", "nb04_run_id", "project_config_version",
                 "feature_registry_version", "created_at_utc"}
mega_cols_to_join = [c for c in mega_feature_table.columns if c not in _meta_exclude]

training_supervision_table = mega_feature_table[mega_cols_to_join].merge(
    union_backed[[
        "union_id", "split_id", "split_role", "supervision_status",
        "target_binary", "target_source", "sample_weight",
        "match_status", "annotation_id", "gt_distance_px", "gt_label",
    ]],
    on="union_id",
    how="left",
)

# Fill unsupervised candidates (outside annotated territory)
training_supervision_table["supervision_status"] = training_supervision_table["supervision_status"].fillna("excluded")
training_supervision_table["target_source"]      = training_supervision_table["target_source"].fillna("outside_supervision_territory")
training_supervision_table["sample_weight"]      = training_supervision_table["sample_weight"].fillna(0.0)
training_supervision_table["target_binary"]      = pd.to_numeric(training_supervision_table["target_binary"], errors="coerce")

log(f"training_supervision_table: {len(training_supervision_table):,} rows × {len(training_supervision_table.columns)} cols")
log(f"Class balance: {n_pos_w:,} positive / {n_neg_w:,} negative  (total supervised {n_total_w:,})")
log(f"Positive rate: {n_pos_w / max(n_total_w, 1):.4f}")

log("\nPer-split class distribution:")
_sup = training_supervision_table[training_supervision_table["target_binary"].notna()]
_split_summary = (
    _sup.groupby("split_role")["target_binary"]
    .agg([("n_positive", "sum"), ("n_total", "count"), ("pos_rate", "mean")])
)
_split_summary["n_negative"] = _split_summary["n_total"] - _split_summary["n_positive"]
display(_split_summary[["n_positive", "n_negative", "n_total", "pos_rate"]])

In [ ]:
# -------------------------------------------------------------------
# SUPERVISION TABLE VALIDATION  (§3.9)
# -------------------------------------------------------------------
required_supervision_cols = {
    "union_id", "split_id", "supervision_status",
    "target_binary", "target_source", "sample_weight",
}
missing_sup = sorted(required_supervision_cols - set(training_supervision_table.columns))
assert not missing_sup, f"training_supervision_table missing columns: {missing_sup}"
assert training_supervision_table["union_id"].is_unique, "training_supervision_table: must be one row per union_id"

allowed_sup_status = {"included", "excluded"}
bad_sup = sorted(set(training_supervision_table["supervision_status"]) - allowed_sup_status)
assert not bad_sup, f"Invalid supervision_status values: {bad_sup}"

included = training_supervision_table["supervision_status"] == "included"
bad_targets = training_supervision_table[included & ~training_supervision_table["target_binary"].isin([0.0, 1.0])]
assert len(bad_targets) == 0, f"{len(bad_targets)} included rows have non-binary target"

excluded_nonzero = training_supervision_table[
    (training_supervision_table["supervision_status"] == "excluded") &
    (training_supervision_table["sample_weight"] != 0.0)
]
assert len(excluded_nonzero) == 0, "Excluded rows must have sample_weight == 0.0"

# Row count must match mega_feature_table
assert len(training_supervision_table) == len(mega_feature_table), (
    f"Row count mismatch: supervision_table={len(training_supervision_table)} vs mega={len(mega_feature_table)}"
)

log("training_supervision_table validation passed.")
display(training_supervision_table[["union_id", "crop_id", "supervision_status", "target_binary", "target_source", "split_role", "sample_weight"]].head(10))

In [ ]:
# -------------------------------------------------------------------
# QA SUMMARIES
# -------------------------------------------------------------------
log("=" * 90)
log("QA SUMMARIES")

# Per-well class distribution
log("\nPer-well class distribution:")
_sup = training_supervision_table[training_supervision_table["target_binary"].notna()]
display(
    _sup.groupby("well_id")["target_binary"]
    .agg(n_pos="sum", n_total="count", pos_rate="mean")
)

# Per-crop class distribution + warnings
log("\nPer-crop class distribution:")
crop_class = _sup.groupby("crop_id")["target_binary"].agg(n_pos="sum", n_total="count", pos_rate="mean")
display(crop_class)
for cid, row in crop_class.iterrows():
    if row["n_pos"] == 0:
        log(f"  [warn] crop {cid}: 0 positives (all negative)")
    if row["pos_rate"] > 0.5:
        log(f"  [warn] crop {cid}: pos_rate={row['pos_rate']:.2f} > 50% — unusual")

# Split leakage check
crop_splits = training_supervision_table.dropna(subset=["split_role"]).groupby("crop_id")["split_role"].nunique()
leaky = crop_splits[crop_splits > 1]
assert len(leaky) == 0, f"Split leakage: {len(leaky)} crops appear in multiple splits: {leaky.index.tolist()}"
log("No split leakage ✓")

# Feature separability: Cohen's d between positive and negative classes
log("\nTop features by class separation (Cohen's d):")
_meta_cols = {
    "union_id", "split_id", "split_role", "supervision_status", "target_binary",
    "target_source", "sample_weight", "match_status", "annotation_id",
    "gt_distance_px", "gt_label", "dataset_id", "well_id", "crop_id", "cluster_id",
    "well_y_px", "well_x_px", "union_centroid_well_y_px", "union_centroid_well_x_px",
    "in_annotated_crop", "annotation_coverage_status", "is_completed_for_negatives",
    "eligible_for_negative_supervision", "coverage_allows_negative",
}
feature_cols_in_sup = [
    c for c in training_supervision_table.columns
    if c not in _meta_cols
    and pd.api.types.is_numeric_dtype(training_supervision_table[c])
]
separability = []
for col in feature_cols_in_sup:
    pos_v = _sup.loc[_sup["target_binary"] == 1, col].dropna()
    neg_v = _sup.loc[_sup["target_binary"] == 0, col].dropna()
    if len(pos_v) > 1 and len(neg_v) > 1:
        pooled_std = math.sqrt(
            (pos_v.var() * len(pos_v) + neg_v.var() * len(neg_v))
            / max(len(pos_v) + len(neg_v) - 2, 1)
        )
        d = (float(pos_v.mean()) - float(neg_v.mean())) / max(pooled_std, 1e-8)
        separability.append({"feature": col, "cohens_d": d, "abs_d": abs(d)})
if separability:
    sep_df = pd.DataFrame(separability).sort_values("abs_d", ascending=False)
    display(sep_df.head(20))

In [ ]:
# -------------------------------------------------------------------
# OPTIONAL QA OVERLAYS
# Renders per-crop overlays for worst-FN then worst-precision crops.
# Degrades gracefully if image paths are not resolvable from crop registry.
# -------------------------------------------------------------------
def try_find_crop_base_image(crop_row: pd.Series):
    candidate_cols = [c for c in crop_row.index if "path" in str(c).lower() or "file" in str(c).lower()]
    for c in candidate_cols:
        v = crop_row[c]
        if isinstance(v, str) and len(v.strip()):
            p = Path(v)
            if p.exists():
                return p
    return None

def read_image_2d(path) -> np.ndarray:
    arr = np.asarray(tifffile.imread(str(path)))
    arr = np.squeeze(arr)
    if arr.ndim == 3:
        arr = arr.max(axis=0)
    if arr.ndim != 2:
        arr = arr.reshape(arr.shape[-2], arr.shape[-1])
    return arr.astype(np.float32, copy=False)

def choose_overlay_crops(summary_df: pd.DataFrame, max_n: int) -> pd.DataFrame:
    if len(summary_df) == 0:
        return summary_df
    return (
        summary_df
        .sort_values(
            ["unmatched_truth_false_negative", "unmatched_candidate_negative", "matched_positive"],
            ascending=[False, False, False]
        )
        .head(max_n)
        .reset_index(drop=True)
    )

COLOR_MAP = {
    "matched_positive":               "lime",
    "unmatched_candidate_negative":   "red",
    "excluded_uncertain":             "gold",
    "unmatched_truth_false_negative": "cyan",
}

overlay_crops = choose_overlay_crops(match_counts_pivot, CFG["qa_max_overlay_crops"])
crop_status_idx = crop_status.set_index(["dataset_id", "well_id", "crop_id"])

if len(overlay_crops) == 0:
    log("No overlay crops to render.")
else:
    log(f"Rendering up to {len(overlay_crops)} QA overlay crops ...")
    for _, row in overlay_crops.iterrows():
        ckey = (row["dataset_id"], row["well_id"], row["crop_id"])
        if ckey not in crop_status_idx.index:
            continue
        crop_row = crop_status_idx.loc[ckey]
        img_path = try_find_crop_base_image(crop_row)
        if img_path is None:
            log(f"  [overlay skip] no resolvable image for crop={row['crop_id']}")
            continue
        try:
            full = read_image_2d(img_path)
        except Exception as exc:
            log(f"  [overlay skip] crop={row['crop_id']}: {type(exc).__name__}: {exc}")
            continue

        y0 = int(crop_row["well_ymin_px"]); x0 = int(crop_row["well_xmin_px"])
        y1 = int(crop_row["well_ymax_px"]); x1 = int(crop_row["well_xmax_px"])
        patch = full[y0:y1, x0:x1]
        lo, hi = np.percentile(patch, 1), np.percentile(patch, 99)
        patch_display = np.clip((patch - lo) / max(hi - lo, 1e-8), 0, 1)

        cand_sub = candidate_to_truth_match[
            (candidate_to_truth_match["dataset_id"] == row["dataset_id"]) &
            (candidate_to_truth_match["well_id"] == row["well_id"]) &
            (candidate_to_truth_match["crop_id"] == row["crop_id"])
        ]

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(patch_display, cmap="gray", origin="upper")

        for status, g in cand_sub.groupby("match_status", dropna=False):
            col = COLOR_MAP.get(status, "white")
            g_cand = g[g["union_id"].notna() & np.isfinite(g["candidate_well_y_px"]) & np.isfinite(g["candidate_well_x_px"])]
            if len(g_cand):
                ax.scatter(
                    g_cand["candidate_well_x_px"] - x0,
                    g_cand["candidate_well_y_px"] - y0,
                    s=28, facecolors="none", edgecolors=col,
                    linewidths=1.3, alpha=CFG["qa_point_alpha"], label=f"{status} (cand)"
                )
            g_truth = g[g["annotation_id"].notna() & np.isfinite(g["gt_click_well_y_px"]) & np.isfinite(g["gt_click_well_x_px"])]
            if len(g_truth):
                ax.scatter(
                    g_truth["gt_click_well_x_px"] - x0,
                    g_truth["gt_click_well_y_px"] - y0,
                    s=40, marker="x", c=col,
                    linewidths=1.5, alpha=CFG["qa_point_alpha"], label=f"{status} (truth)"
                )

        ax.legend(loc="upper right", fontsize=7, framealpha=0.7)
        ax.set_title(
            f"{row['well_id']} | {row['crop_id'][-12:]}\n"
            f"pos={int(row.get('matched_positive',0))}  "
            f"neg={int(row.get('unmatched_candidate_negative',0))}  "
            f"fn={int(row.get('unmatched_truth_false_negative',0))}  "
            f"excl={int(row.get('excluded_uncertain',0))}",
            fontsize=9
        )
        ax.set_axis_off()
        fig.tight_layout()
        out_png = REPORT_DIR / f"{RUN_ID}_{row['crop_id']}_overlay.png"
        fig.savefig(out_png, dpi=CFG["qa_fig_dpi"], bbox_inches="tight")
        plt.show()
        plt.close(fig)
        log(f"  Saved: {out_png.name}")

In [ ]:
# -------------------------------------------------------------------
# PERSIST OUTPUTS
# -------------------------------------------------------------------
log("=" * 90)
log("PERSISTING OUTPUTS")

candidate_to_truth_match_path  = None
training_supervision_table_path = None
truth_evaluation_summary_path  = None
splits_path                    = None
matching_timing_path           = None
crop_summary_path              = None

if CFG["write_candidate_to_truth_match"]:
    candidate_to_truth_match_path = safe_to_parquet(
        candidate_to_truth_match,
        TABLES_DIR / f"{RUN_ID}_candidate_to_truth_match.parquet"
    )

if CFG["write_training_supervision_table"]:
    training_supervision_table_path = safe_to_parquet(
        training_supervision_table,
        TABLES_DIR / f"{RUN_ID}_training_supervision_table.parquet"
    )

if CFG["write_truth_evaluation_summary"]:
    truth_evaluation_summary_path = safe_to_parquet(
        truth_evaluation_summary if len(truth_evaluation_summary) else pd.DataFrame(columns=[
            "annotation_id", "dataset_id", "well_id", "crop_id", "label",
            "truth_well_y_px", "truth_well_x_px", "match_status",
            "matched_union_id", "nearest_union_distance_px", "matching_registry_version",
        ]),
        TABLES_DIR / f"{RUN_ID}_truth_evaluation_summary.parquet"
    )

if CFG["write_splits"]:
    splits_path = safe_to_parquet(
        splits_df,
        TABLES_DIR / f"{RUN_ID}_splits.parquet"
    )

matching_timing_path = safe_to_parquet(
    matching_timing_df,
    TABLES_DIR / f"{RUN_ID}_matching_timing.parquet"
)
crop_summary_path = safe_to_parquet(
    crop_match_summary,
    TABLES_DIR / f"{RUN_ID}_crop_match_summary.parquet"
)

manifest = {
    "run_id": RUN_ID,
    "notebook_name": "05_truth_matching.ipynb",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_root": str(REPO_ROOT),
    "matching_algorithm": CFG["matching_algorithm"],
    "matching_registry_version": CFG["matching_registry_version"],
    "truth_match_radius_px": CFG["truth_match_radius_px"],
    "uncertain_exclusion_policy": CFG["uncertain_exclusion_policy"],
    "uncertain_halo_radius_px": CFG["uncertain_halo_radius_px"],
    "split_registry_version": CFG["split_registry_version"],
    "sample_weight_strategy": CFG["sample_weight_strategy"],
    "counts": {
        "n_matched_positive":    n_pos,
        "n_negative":            n_neg,
        "n_false_negative":      n_fn,
        "n_excluded_uncertain":  n_excl,
        "n_supervised_total":    int(n_total_w),
        "detection_recall":      float(n_pos / max(n_pos + n_fn, 1)),
        "positive_rate":         float(n_pos / max(n_pos + n_neg, 1)),
        "n_splits_train":        int((splits_df["split_role"] == "train").sum()),
        "n_splits_cal":          int((splits_df["split_role"] == "calibration").sum()),
        "n_splits_test":         int((splits_df["split_role"] == "test").sum()),
        "w_pos":                 float(w_pos),
        "w_neg":                 float(w_neg),
    },
    "inputs": INPUT_PATHS,
    "outputs": {
        "candidate_to_truth_match":   str(candidate_to_truth_match_path) if candidate_to_truth_match_path else None,
        "training_supervision_table": str(training_supervision_table_path) if training_supervision_table_path else None,
        "truth_evaluation_summary":   str(truth_evaluation_summary_path) if truth_evaluation_summary_path else None,
        "splits":                     str(splits_path) if splits_path else None,
        "matching_timing":            str(matching_timing_path),
        "crop_match_summary":         str(crop_summary_path),
        "report_dir":                 str(REPORT_DIR),
    },
}
manifest_path = write_json(manifest, MANIFEST_DIR / f"{RUN_ID}_truth_matching_manifest.json")

for k, v in manifest["outputs"].items():
    log(f"  {k}: {Path(v).name if v else None}")
log(f"  manifest: {manifest_path.name}")

In [ ]:
# -------------------------------------------------------------------
# EXIT CHECKS
# -------------------------------------------------------------------
log("=" * 90)
log("EXIT CHECKS")

# ── candidate_to_truth_match (§3.7) ───────────────────────────────────────
required_match_cols = {
    "match_id", "union_id", "annotation_id", "match_status", "gt_label",
    "gt_distance_px", "matching_algorithm", "truth_match_radius_px",
    "matching_registry_version",
}
missing = sorted(required_match_cols - set(candidate_to_truth_match.columns))
assert not missing, f"candidate_to_truth_match missing: {missing}"
assert candidate_to_truth_match["match_id"].is_unique, "match_id must be unique"

_pos_check = candidate_to_truth_match[candidate_to_truth_match["match_status"] == "matched_positive"]
assert _pos_check["union_id"].is_unique,      "One-to-one violated: union_id repeated in matched_positive"
assert _pos_check["annotation_id"].is_unique, "One-to-one violated: annotation_id repeated in matched_positive"

# ── splits (§3.10) ────────────────────────────────────────────────────────
required_split_cols = {
    "split_id", "dataset_id", "well_id", "crop_id",
    "spatial_block_id", "split_role", "split_registry_version",
}
missing = sorted(required_split_cols - set(splits_df.columns))
assert not missing, f"splits missing: {missing}"
assert splits_df["crop_id"].is_unique, "Split leakage: duplicate crop_id in splits"
assert set(splits_df["split_role"]).issubset({"train", "calibration", "test"})

# ── training_supervision_table (§3.9) ─────────────────────────────────────
required_sup_cols = {
    "union_id", "split_id", "supervision_status",
    "target_binary", "target_source", "sample_weight",
}
missing = sorted(required_sup_cols - set(training_supervision_table.columns))
assert not missing, f"training_supervision_table missing: {missing}"
assert training_supervision_table["union_id"].is_unique, "training_supervision_table: duplicate union_id"

_sup_included = training_supervision_table[
    training_supervision_table["supervision_status"] == "included"
]
assert _sup_included["union_id"].is_unique, "Duplicate union_id in included supervision rows"
assert len(training_supervision_table) == len(mega_feature_table), (
    f"Row count mismatch: supervision={len(training_supervision_table)} vs mega={len(mega_feature_table)}"
)

# All supervised union_ids exist in mega_feature_table
mega_uids = set(mega_feature_table["union_id"])
sup_uids  = set(training_supervision_table["union_id"].dropna())
orphans   = sup_uids - mega_uids
assert len(orphans) == 0, f"{len(orphans)} supervised union_ids not in mega_feature_table"

log(f"\n{'=' * 90}")
log(f"NOTEBOOK 05 COMPLETE")
log(f"  candidate_to_truth_match:     {len(candidate_to_truth_match):,} rows")
log(f"    matched_positive:               {n_pos:,}")
log(f"    unmatched_candidate_negative:   {n_neg:,}")
log(f"    unmatched_truth_false_negative: {n_fn:,}")
log(f"    excluded_uncertain:             {n_excl:,}")
log(f"  detection recall (pre-classifier): {n_pos / max(n_pos + n_fn, 1):.3f}")
log(f"  positive rate in supervision:      {n_pos / max(n_pos + n_neg, 1):.4f}")
log(f"  splits: {len(splits_df)} crops → "
    f"{int((splits_df['split_role'] == 'train').sum())} train / "
    f"{int((splits_df['split_role'] == 'calibration').sum())} cal / "
    f"{int((splits_df['split_role'] == 'test').sum())} test")
log(f"  training_supervision_table:   {len(training_supervision_table):,} rows × {len(training_supervision_table.columns)} cols")
log(f"{'=' * 90}")